In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"]="false"

import pathlib
from functools import partial

import time
from tqdm.notebook import tqdm
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
mpl.rcParams['text.usetex'] = True
mpl.rcParams['text.latex.preamble']=r"\usepackage{bm}"

In [ ]:
import jax
import jax.numpy as jnp
import equinox as eqx
import diffrax

import gpjax as gpx

In [ ]:
cols = mpl.rcParams["axes.prop_cycle"].by_key()["color"]

In [ ]:
from admpe.evaluation.model_wrapping import get_wrapped_model_and_env
from admpe.data_management import (
    list_modeling_results, load_modeling_result, provide_modeling_results_overview, load_excitation_result
)
from admpe.evaluation.modeling_evaluation import (
    evaluate_model, update_modeling_df, load_corresponding_excitation_result, evaluate_model_prediction, evaluate_model_rollout,
)
from admpe.evaluation.modeling_visualization import visualize_model_prediction, visualize_model_rollouts

In [ ]:
results = list_modeling_results(
    system_name=None,
    model_type="neural_euler",
    status="completed",
    exp_name="run1"
)
print(len(results))
results

In [ ]:
provide_modeling_results_overview(results)

In [ ]:
load_modeling_result(results[0]["path"])

In [ ]:
results = list_modeling_results(
    system_name=None,
    model_type="neural_euler",
    status="completed",
    exp_name="run1"
)

## basic model performance metrics:

#### prediction:

In [ ]:
result = load_modeling_result(results[5]["path"])
wrapped_model, wrapped_env = get_wrapped_model_and_env(result)
evaluate_model_prediction(wrapped_model, wrapped_env)

In [ ]:
result = load_modeling_result(results[6]["path"])
wrapped_model, wrapped_env = get_wrapped_model_and_env(result)
evaluate_model_prediction(wrapped_model, wrapped_env)

In [ ]:
visualize_model_prediction(20, wrapped_model, wrapped_env)

In [ ]:
excitation_result = load_corresponding_excitation_result(result)
excitation_result["estimator"].visualize()
plt.show()

plt.plot(excitation_result["observations"])
plt.show()

#### rollouts

In [ ]:
results = list_modeling_results(
    system_name=None,
    model_type="neural_euler",
    status="completed",
    exp_name="run1"
)

In [ ]:
modeling_result = load_modeling_result(results[5]["path"])
excitation_result = load_corresponding_excitation_result(modeling_result)

wrapped_model, wrapped_env = get_wrapped_model_and_env(modeling_result)
evaluate_model_rollout(
    wrapped_model,
    observations=excitation_result["observations"][None],
    actions=excitation_result["actions"][None],
)

In [ ]:
modeling_result = load_modeling_result(results[6]["path"])
excitation_result = load_corresponding_excitation_result(modeling_result)

wrapped_model, wrapped_env = get_wrapped_model_and_env(modeling_result)
evaluate_model_rollout(
    wrapped_model,
    observations=excitation_result["observations"][None],
    actions=excitation_result["actions"][None],
)

In [ ]:
visualize_model_rollouts(
    wrapped_model,
    observations=jnp.concatenate([
        excitation_result["observations"][None, 1000:2001],
        excitation_result["observations"][None, 2000:3001],
        excitation_result["observations"][None, 3000:4001],
    ], axis=0),
    actions=jnp.concatenate([
        excitation_result["actions"][None, 1000:2000],
        excitation_result["actions"][None, 2000:3000],
        excitation_result["actions"][None, 3000:4000],
    ], axis=0),
)